# Player match history and product export

Build chronological batting and bowling history tables from audited match events and verified Stage 09 partial NWC. Stage 09 uses the four models selected by Stage 08 across feature groups using validation Log Loss. The scope is the recorded test matches, not a complete career history.

This stage exports data only: no training, prediction or attribution. The API and frontend are separate consumers.

Package location: `outputs/10_results_and_visualisation/player_history_v1/`. See `docs/player_history_export.md` for field definitions and verification.


In [11]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'src' / 'player_history_export.py').is_file():
    raise FileNotFoundError('Open from T20_Project_3.0 or its notebooks directory.')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
import player_history_export as stage
importlib.reload(stage)
RUN_EXPORT = False
GENDER = 'female'
PLAYER_ID = None  # Set a stable player ID from the catalogue below.
if RUN_EXPORT:
    stage.run(ROOT)
PACKAGE = stage.verified_output(ROOT)
payload = json.loads((PACKAGE / 'history.json').read_text(encoding='utf-8'))
checks = json.loads((PACKAGE / 'verification.json').read_text(encoding='utf-8'))
print('Verified package:', PACKAGE)
print('Scope:', payload['scope_label'])
print('Stage 09 attribution:', payload['upstream_attribution_id'])
print('Stage 08 evaluation:', payload['upstream_evaluation_id'])


Verified package: D:\T20_Project\T20_Project_3.0\outputs\10_results_and_visualisation\player_history_v1\snapshots\8e9f523ce24dc6af3a9d0f393469720e8911aec76848d0ffaf3777d0da37a21f
Scope: Current recorded test matches; not complete player careers
Stage 09 attribution: f6a25bf2480b950fe31e69672273d7a4c06807f72e6efce1098ba6261d48baae
Stage 08 evaluation: 31bd1c42bb6e0ec84f9824da5fe496de9fc7e0680f32c96bc2db445e30fa3bea


## Coverage and validation

Counts are restricted to this package. A listed player is not automatically a confirmed match appearance. Verification checks all source events, all player statistics and all Stage 09 role contributions.


In [12]:
display(pd.DataFrame(payload['counts_by_gender']).T)
display(pd.DataFrame([checks]))
players = pd.DataFrame(payload['players'])
catalogue = players[players.gender.eq(GENDER)].sort_values(['name', 'player_id'])
display(catalogue[['player_id', 'name', 'confirmed_match_count', 'listed_match_count']].head(20))


,matches,players,player_match_rows,date_from,date_to
female,87,230,1914,2025-01-31,2026-08-04
male,141,312,3102,2024-12-05,2026-07-26


,status,player_match_rows,verified_statistics,verified_nwc_roles,verified_team_rows,max_team_closure_error,non_boundary_scoring_events,innings_checked,event_rows,source_json_alignment,training_fits,prediction_calls
0,passed,5016,"[runs, balls_faced, dismissals, legal_balls_bo...",6402,456,8.326673e-16,3,456,53850,all events matched by match/innings/sequence a...,0,0


,player_id,name,confirmed_match_count,listed_match_count
195,dcf0e8f4,A Bosch,4,4
229,ffd5fcd8,A Canning,8,8
118,910dd54e,A Capsey,19,19
81,66498395,A Dalzell,4,4
182,d2b1f515,A Dercksen,18,18
160,bc969efb,A Gardner,7,7
178,ccaa3434,A Hlubi,4,4
208,e8fa03ed,A Hunter,14,14
6,0d518f89,A Khaka,17,17
103,83558266,A King,7,7


## Player selection and chronological history

Choose a gender and stable player ID in the setup cell. With no ID, the first player in the displayed catalogue is selected. Both tables show all recorded matches for that player in ascending date order, including explicit did-not-bat and did-not-bowl states.

NWC is the allocated part only. It is displayed in percentage points alongside role-event coverage. Missing values remain missing, not zero. A non-striker-only appearance can have no striker-role NWC.


In [13]:
if catalogue.empty:
    raise ValueError('No players for selected gender.')
selected_id = PLAYER_ID if PLAYER_ID is not None else catalogue.iloc[0].player_id
selected = [r for r in payload['rows'] if r['gender'] == GENDER and r['player_id'] == selected_id]
if not selected:
    raise ValueError('Player ID is not available in this gender and dataset.')
print('Player:', selected[-1]['player_name'], '| ID:', selected_id)
print('Recorded match rows:', len(selected))
def history_view(role, fields):
    table = stage.role_table(selected, role)
    table['partial_nwc_pp'] = table.nwc_partial_nwc * 100
    return table[['date', 'match_id', 'team_name', 'opponent_name'] + fields +
                 ['partial_nwc_pp', 'nwc_status', 'nwc_role_events', 'nwc_allocated_events', 'nwc_allocation_coverage']]


Player: A Bosch | ID: dcf0e8f4
Recorded match rows: 4


## Batting history

Fours and sixes exclude source-marked non-boundaries. The table reports dismissal count and recorded exit status separately. Dismissal belongs to the player who exited, including a non-striker run-out.


In [14]:
display(history_view('batting', ['batted', 'runs', 'balls_faced', 'strike_rate', 'fours', 'sixes', 'dismissals', 'dismissal']))


,date,match_id,team_name,opponent_name,batted,runs,balls_faced,strike_rate,fours,sixes,dismissals,dismissal,partial_nwc_pp,nwc_status,nwc_role_events,nwc_allocated_events,nwc_allocation_coverage
0,2026-03-25,1491709,South Africa,New Zealand,True,9.0,8.0,112.500000,2.0,0.0,1.0,caught,0.077720,available,9,7,0.777778
1,2026-04-17,1527253,South Africa,India,True,8.0,6.0,133.333333,0.0,1.0,1.0,lbw,0.498171,available,8,6,0.750000
2,2026-04-19,1527254,South Africa,India,False,NaN,NaN,NaN,NaN,NaN,NaN,did not bat,NaN,not_participated,0,0,NaN
3,2026-04-27,1527257,South Africa,India,True,8.0,11.0,72.727273,0.0,0.0,1.0,bowled,-1.590121,available,11,11,1.000000


## Bowling history

Overs are six-ball equivalents of actual legal balls, not decimal numbers. Conceded runs exclude byes, leg-byes and separate penalties. Run-outs are not credited as bowler wickets.


In [15]:
display(history_view('bowling', ['bowled', 'overs_display', 'legal_balls_bowled', 'runs_conceded', 'wickets', 'economy', 'dot_balls']))


,date,match_id,team_name,opponent_name,bowled,overs_display,legal_balls_bowled,runs_conceded,wickets,economy,dot_balls,partial_nwc_pp,nwc_status,nwc_role_events,nwc_allocated_events,nwc_allocation_coverage
0,2026-03-25,1491709,South Africa,New Zealand,False,None,None,None,None,None,None,NaN,not_participated,0,0,None
1,2026-04-17,1527253,South Africa,India,False,None,None,None,None,None,None,NaN,not_participated,0,0,None
2,2026-04-19,1527254,South Africa,India,False,None,None,None,None,None,None,NaN,not_participated,0,0,None
3,2026-04-27,1527257,South Africa,India,False,None,None,None,None,None,None,NaN,not_participated,0,0,None


## Product handoff

Use `current.json` to resolve the verified package. `history.json` contains stable player IDs, aliases, team and match references and both role histories. The two CSV files are inspection projections. API and frontend adapters must preserve NWC status, scope and units; the prototype's R0 data contract is not this Schema 2 contract.

The package contains partial operational contributions, not complete causal player value. First-innings boundary and other team residuals are not allocated to players. No complete-career or live-prediction coverage is claimed.
